In [1]:
import os
import warnings
from dotenv import load_dotenv

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
warnings.filterwarnings("ignore")
load_dotenv()

True

In [2]:
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate,
                                    PromptTemplate,
                                    ChatMessagePromptTemplate,
                                    ChatPromptTemplate)
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
base_url="http://localhost:11434"
model='llama3.2:3b'


llm=ChatOllama(base_url=base_url,model=model)
llm.invoke('hi')

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-22T17:57:30.545538Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2842839625, 'load_duration': 2361444833, 'prompt_eval_count': 26, 'prompt_eval_duration': 297483500, 'eval_count': 8, 'eval_duration': 182202666, 'message': Message(role='assistant', content='', images=None, tool_calls=None)}, id='run--0fa1a5bf-0f2c-46ac-8f01-3b8730851bfe-0', usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

### Define Tools

In [5]:
from langchain_tavily import TavilySearch
@tool
def search(query: str) -> str:
    """ Realtime and latest information
    """

    search = TavilySearch(max_results=5,
    include_answer=True,
    include_raw_content=True,
    # include_images=False,
    # include_image_descriptions=False,
    search_depth="advanced",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
    )
    return search.invoke(query)

In [6]:
from langchain_ollama import OllamaEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [7]:
embeddings = OllamaEmbeddings(model='nomic-embed-text',base_url="http://localhost:11434")
db_name="/Users/mdmehranabul/Desktop/Learning/Langchain_Ollama/Vector_Store_and_Retrieval/health_supplements"
vector_store=FAISS.load_local(db_name,embeddings, allow_dangerous_deserialization=True)

In [9]:
retriever = vector_store.as_retriever(search_type="similarity",
                                      search_kwargs={'k':3})

In [10]:
@tool
def health_supplement(query: str) -> str:
    """ Search for information about Health Supplements

    Args:
        query: The search query
    """
    response = retriever.invoke(query)
    return response

In [11]:
health_supplement.invoke("what is the best supplement for muscle gain?")

[Document(id='a232e405-6db3-4448-a5bf-6593fa5c8b7b', metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf', 'file_path': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-10-21T11:38:50+00:00', 'trapped': '', 'modDate': 'D:20241021113850Z', 'creationDate': '', 'page': 0}, page_content='acids than traditional protein sources. Its numerous benefits have made it a popular choice\nfor snacks and drinks among consumers [3]. Another widely embraced supplement is\ncaffeine, which is found in many sports and food supplements. Caffeine reduces perceived\neffort, minimizes fatigue and pain, and proves to be effective for endurance and high-\nintensity activities, which is the choice of consumers [4].\nCreatine monohydrate is another well-known supplement 

### Agent Creation

In [12]:
from langchain import hub

from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

In [13]:
prompt = hub.pull("hwchase17/openai-functions-agent")

In [15]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [16]:
tools =[search, health_supplement]
agent = create_tool_calling_agent(llm, tools, prompt)

In [17]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [27]:
# question = "What is the best supplement for muscle gain?"
question = "How is the weather in New York?"
response = agent_executor.invoke({'input' : question})



> Entering new AgentExecutor chain...

Invoking: `search` with `{'query': 'New York weather'}`


{'query': 'New York weather', 'follow_up_questions': None, 'answer': 'In September 2025, New York City will have warm temperatures, ranging from 68°F to 81°F, with occasional rain and thunderstorms. The weather will be mostly cloudy to partly cloudy. Expect a few sunny days interspersed with rain.', 'images': [], 'results': [{'url': 'https://en.climate-data.org/north-america/united-states-of-america/new-york/new-york-1091/t/september-9/', 'title': 'Weather New York in September 2025: Temperature & Climate', 'content': '| 24. September | 19 °C | 66 °F | 23 °C | 74 °F | 15 °C | 58 °F | 21 °C | 69 °F | 2.1 mm | 0.1 inch. |\n| 25. September | 18 °C | 65 °F | 23 °C | 73 °F | 15 °C | 58 °F | 20 °C | 69 °F | 2.4 mm | 0.1 inch. |\n| 26. September | 18 °C | 65 °F | 22 °C | 72 °F | 15 °C | 58 °F | 20 °C | 69 °F | 2.2 mm | 0.1 inch. |\n| 27. September | 18 °C | 65 °F | 23 °C | 73 °F | 15 °C | 59 °F 

In [ ]:
print(response['output'])

The current weather forecast for New York City is:

* Today:
	+ High temperature: 79°F
	+ Low temperature: 68°F
* Extended forecast:
	+ Hourly forecast available
	+ 7-day forecast available
	+ 10-day forecast available
	+ 14-day forecast available
	+ 30-day forecast available

Additionally, the weather forecast for nearby cities is also provided:

* Washington, D.C.: +68°F
* Albany: +64°F
* Linden: +72°F
* Yonkers: +68°F
* Mount Vernon: +68°F
* Paterson: +68°F
* Livingston: +68°F
* New Rochelle: +70°F
* Summit: +68°F
* Rockville Centre: +72°F
* Union: +72°F
* Clifton: +70°F
* Jersey City: +73°F
* Bayonne: +72°F
* North Bergen: +72°F
* East New York: +72°F
* Exchange Place North: +73°F
* Rochelle Park: +70°F

Please note that these forecasts are subject to change and may not be accurate at the time of reading.
